In [1]:
import sqlite3
import pandas as pd
import requests
import time

In [68]:
%load_ext autoreload
%autoreload 2

In [2]:
url = "http://universities.hipolabs.com/search?country=Brazil"

In [3]:
response = requests.get(url)
response.raise_for_status()
universities = response.json()

In [4]:
universities[0]

{'name': 'Universidade Comunitária da Região de Chapecó - Unochapecó',
 'web_pages': ['https://unochapeco.edu.br'],
 'domains': ['unochapeco.edu.br'],
 'country': 'Brazil',
 'alpha_two_code': 'BR',
 'state-province': None}

In [5]:
db_universities = sqlite3.connect('universities.db')

c = db_universities.cursor()

In [6]:
c.execute('''
CREATE TABLE IF NOT EXISTS universidades_brasil
(id INTEGER PRIMARY KEY AUTOINCREMENT,
name TEXT,
country TEXT,
state_province TEXT,
web_pages TEXT,
domains TEXT);''')


In [7]:
for uni in universities:
    c.execute('''
    INSERT INTO universidades_brasil (name, country, state_province, web_pages, domains)
    VALUES (?, ?, ?, ?, ?);''', 
    (uni.get('name'), 
     uni.get('country'), 
     uni.get('state-province'), 
     ', '.join(uni.get('web_pages', [])), 
     ', '.join(uni.get('domains', []))
     )
     )

In [8]:
db_universities.commit()
db_universities.close()

In [9]:
conn = sqlite3.connect('universities.db')

In [16]:
def sq(str, conn=conn):
    return pd.read_sql('''{}'''.format(str), conn)

In [17]:
sq('''SELECT * FROM sqlite_master WHERE type='table' ''')

,type,name,tbl_name,rootpage,sql
0,table,universidades_brasil,universidades_brasil,2,CREATE TABLE universidades_brasil\n(id INTEGER...
1,table,sqlite_sequence,sqlite_sequence,3,"CREATE TABLE sqlite_sequence(name,seq)"


In [20]:
sq('''SELECT * FROM universidades_brasil''')

,id,name,country,state_province,web_pages,domains
0,1,Universidade Comunitária da Região de Chapecó ...,Brazil,NaN,https://unochapeco.edu.br,unochapeco.edu.br
1,2,"Centro Universitário de Brasília, UNICEUB",Brazil,NaN,https://www.uniceub.br,"sempreceub.com, uniceub.br"
2,3,Centro Universitário Barao de Maua,Brazil,NaN,http://www.baraodemaua.br/,baraodemaua.br
3,4,Universidade Braz Cubas,Brazil,NaN,http://www.brazcubas.br/,brazcubas.br
4,5,Universidade Candido Mendes,Brazil,NaN,http://www.candidomendes.br/,candidomendes.br
...,...,...,...,...,...,...
375,376,Centro Universitário Álvares Penteado - FECAP,Brazil,São Paulo,https://www.fecap.br/,fecap.br
376,377,Centro Universitário Facens - UniFACENS,Brazil,São Paulo,https://facens.br/,facens.br
377,378,Centro Universitário Antônio Eufrásio de Toled...,Brazil,Presidente Prudente,https://toledoprudente.edu.br/,toledoprudente.edu.br
378,379,Faculdade Impacta - IMPACTA,Brazil,São Paulo,https://www.impacta.edu.br/,impacta.edu.br


In [21]:
sq('''SELECT * FROM universidades_brasil WHERE name LIKE '%Pernambuco%' ''')

,id,name,country,state_province,web_pages,domains
0,84,Universidade Federal de Pernambuco,Brazil,None,http://www.ufpe.br/,ufpe.br
1,91,Universidade Federal Rural de Pernambuco,Brazil,None,http://www.ufrpe.br/,ufrpe.br
2,115,Universidade Católica de Pernambuco,Brazil,None,http://www.unicap.br/,unicap.br
3,157,Universidade de Pernambuco,Brazil,None,http://www.upe.br/,upe.br
4,176,Polytechnic University of Pernambuco,Brazil,None,https://upe.poli.br,"upe.poli.br, ecomp.poli.br, poli.br"
5,274,Universidade Federal de Pernambuco,Brazil,None,http://www.ufpe.br/,ufpe.br
6,281,Universidade Federal Rural de Pernambuco,Brazil,None,http://www.ufrpe.br/,ufrpe.br
7,305,Universidade Católica de Pernambuco,Brazil,None,http://www.unicap.br/,unicap.br
8,347,Universidade de Pernambuco,Brazil,None,http://www.upe.br/,upe.br
9,366,Polytechnic University of Pernambuco,Brazil,None,https://upe.poli.br,"upe.poli.br, ecomp.poli.br, poli.br"


In [53]:
def universidades_pais(pais: str):

    pais_url = pais.title().replace(' ', '+')
    nome_tabela = pais.lower().replace(' ', '_')
    
    url = f"http://universities.hipolabs.com/search?country={pais_url}"

# Teste de erros 
    max_tentativas = 3
    universities = None
    
    for tentativa in range(max_tentativas):
        try:
            print(f"Baixando dados... (Tentativa {tentativa + 1}/{max_tentativas})")
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            universities = response.json()
            break
            
        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError) as e:
            print("O servidor cortou a conexão no meio do caminho. Tentando novamente em 2 segundos...")
            time.sleep(2)
            
    if not universities:
        print("Erro: Não foi possível baixar os dados após várias tentativas. A API pode estar fora do ar.")
        return


    conn = sqlite3.connect('universities.db')
    c = conn.cursor()

    c.execute(f'''
        CREATE TABLE IF NOT EXISTS universidades_{nome_tabela}
        (id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        country TEXT,
        state_province TEXT,
        web_pages TEXT,
        domains TEXT);''')

    for uni in universities:
        c.execute(f'''
        INSERT INTO universidades_{nome_tabela} (name, country, state_province, web_pages, domains)
        VALUES (?, ?, ?, ?, ?);''',
        (uni.get('name'), 
         uni.get('country'), 
        uni.get('state-province'), 
        ', '.join(uni.get('web_pages', [])), 
        ', '.join(uni.get('domains', []))
     )
     )
        
    conn.commit()
    conn.close()
    print(f"Sucesso! {len(universities)} universidades de {pais.title()} foram salvas na tabela universidades_{nome_tabela}.")

In [54]:
universidades_pais('ireland')

Baixando dados... (Tentativa 1/3)
Sucesso! 27 universidades de Ireland foram salvas na tabela universidades_ireland.


### Modularizando Extração e Carregamento de Dados

In [2]:
from src.extract import Extract

extract = Extract()
br = extract.extract_country('Brazil')
print(type(br))
print(br)

<class 'list'>
[{'name': 'Universidade Comunitária da Região de Chapecó - Unochapecó', 'web_pages': ['https://unochapeco.edu.br'], 'domains': ['unochapeco.edu.br'], 'country': 'Brazil', 'alpha_two_code': 'BR', 'state-province': None}, {'name': 'Centro Universitário de Brasília, UNICEUB', 'web_pages': ['https://www.uniceub.br'], 'domains': ['sempreceub.com', 'uniceub.br'], 'country': 'Brazil', 'alpha_two_code': 'BR', 'state-province': None}, {'name': 'Centro Universitário Barao de Maua', 'web_pages': ['http://www.baraodemaua.br/'], 'domains': ['baraodemaua.br'], 'country': 'Brazil', 'alpha_two_code': 'BR', 'state-province': None}, {'name': 'Universidade Braz Cubas', 'web_pages': ['http://www.brazcubas.br/'], 'domains': ['brazcubas.br'], 'country': 'Brazil', 'alpha_two_code': 'BR', 'state-province': None}, {'name': 'Universidade Candido Mendes', 'web_pages': ['http://www.candidomendes.br/'], 'domains': ['candidomendes.br'], 'country': 'Brazil', 'alpha_two_code': 'BR', 'state-province': N